# SNR-Metrikplots auf dem Testdatensatz

Dieses Notebook erzeugt die SNR-Metrikplots fuer die drei Einzelmodelle sowie die drei Rejection-Ensembles. Alle Ergebnisse werden als PDF in `snr_curves_test/` gespeichert.

In [ ]:
from __future__ import annotations

import copy
import gc
import json
import math
import os
import sys
from collections import defaultdict
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

TRAINING_ROOT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training")
DATA_DIR = TRAINING_ROOT.parent / "DM_time_dataset_creator" / "outputs"
PLOT_DIR = TRAINING_ROOT / "plot" / "snr_curves_test"
METRIC_DIR = PLOT_DIR / "metrics"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

if str(TRAINING_ROOT) not in sys.path:
    sys.path.insert(0, str(TRAINING_ROOT))

from DMTimeShardDataset import DMTimeShardDataset
from moe.checkpoints import load_expert_checkpoint
from moe.train_joint_ensemble import build_joint_model
from training_models import models_htable
from training_utils import label_encoding

DATASET_CFG = {
    "output_dir": str(DATA_DIR),
    "prefix": "B0531+21_59000_48386",
}

SPLIT = "test"
BATCH_SIZE = 512
NUM_WORKERS = 8
RECOMPUTE_METRICS = False

SNR_START = None
SNR_END = 14
Y_MIN = 0.0
INCLUDE_NON_FINITE = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
print(f"PDF output: {PLOT_DIR}")

In [ ]:
BASELINE_DIR = TRAINING_ROOT / "final_checkpoints" / "baseline_rejection_ensemble"
FINETUNE_DIR = TRAINING_ROOT / "final_checkpoints" / "finetune_checkpoints"
JOINT_MOE_CKPT = TRAINING_ROOT / "final_checkpoints" / "joint_cascade_moe_best.pth"

BASELINE_SMALL_CKPT = BASELINE_DIR / "prot-DM_time_binary_classificator_241002_3_GAP-014-0.764-0.740.pth"
BASELINE_MID_CKPT = BASELINE_DIR / "prot-DM_time_binary_classificator_241002_5_GAP-060-0.973-0.948.pth"
BASELINE_LARGE_CKPT = BASELINE_DIR / "prot-DM_time_binary_classificator_resnet18-003-0.993-0.993.pth"
BASELINE_R1_CKPT = BASELINE_DIR / "prot-run_embedding_3_GAP_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker2_trial3-042-0.712-0.635.pth"
BASELINE_R2_CKPT = BASELINE_DIR / "prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker13_trial6-035-0.825-0.842.pth"

FINETUNE_SMALL_CKPT = FINETUNE_DIR / "prot-DM_time_binary_classificator_241002_3_GAP_finetune-004-0.838-0.813.pth"
FINETUNE_MID_CKPT = FINETUNE_DIR / "prot-DM_time_binary_classificator_241002_5_GAP_finetune-019-0.989-0.993.pth"
FINETUNE_LARGE_CKPT = FINETUNE_DIR / "prot-DM_time_binary_classificator_resnet18_finetune-010-0.999-0.993.pth"
FINETUNE_R1_CKPT = FINETUNE_DIR / "prot-run_embedding_r1_conv_mlp_lr1.05e-05_wd0.00e+00_drop0.0_channels64_extraFalse_pool7_hidden64_worker3_trial0-030-0.688-0.618.pth"
FINETUNE_R2_CKPT = FINETUNE_DIR / "prot-run_embedding_r2_conv_mlp_lr4.51e-05_wd0.00e+00_drop0.2_channels64_extraTrue_pool7_hidden128_worker0_trial0-017-0.800-0.834.pth"

BASELINE_R1_THRESHOLD = 0.548808
BASELINE_R2_THRESHOLD = 0.381863
FINETUNE_R1_THRESHOLD = 0.542446
FINETUNE_R2_THRESHOLD = 0.409630
MOE_R1_THRESHOLD = 0.5663751363754272
MOE_R2_THRESHOLD = 0.8648102879524231

all_paths = {
    "baseline f_small": BASELINE_SMALL_CKPT,
    "baseline f_mid": BASELINE_MID_CKPT,
    "baseline f_large": BASELINE_LARGE_CKPT,
    "baseline r1": BASELINE_R1_CKPT,
    "baseline r2": BASELINE_R2_CKPT,
    "finetune f_small": FINETUNE_SMALL_CKPT,
    "finetune f_mid": FINETUNE_MID_CKPT,
    "finetune f_large": FINETUNE_LARGE_CKPT,
    "finetune r1": FINETUNE_R1_CKPT,
    "finetune r2": FINETUNE_R2_CKPT,
    "joint training MoE": JOINT_MOE_CKPT,
}
missing = [f"{name}: {path}" for name, path in all_paths.items() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing checkpoint(s):\n" + "\n".join(missing))
print("Alle Checkpoints gefunden.")

In [ ]:
def make_test_dataset() -> DMTimeShardDataset:
    dataset = DMTimeShardDataset(DATASET_CFG, use_freq_time=True, split=SPLIT)
    dataset.labels = label_encoding(dataset.labels.astype(object))
    return dataset


def build_classifier(model_name: str, mode: str, checkpoint_path: Path, dropout: bool | float = False) -> torch.nn.Module:
    model = models_htable[model_name](256, mode=mode, dropout=dropout, device=DEVICE).to(DEVICE)
    load_expert_checkpoint(model, checkpoint_path, map_location=DEVICE)
    model.eval()
    return model


def make_joint_config(
    small_ckpt: Path,
    mid_ckpt: Path,
    large_ckpt: Path,
    r1_ckpt: Path,
    r2_ckpt: Path,
) -> dict[str, Any]:
    return {
        "dataset": DATASET_CFG,
        "model": {
            "temperature": 1.0,
            "f_small": {
                "model_name": "DM_time_binary_classificator_241002_3_GAP",
                "resolution": 256,
                "mode": "dmt",
                "dropout": False,
                "checkpoint": str(small_ckpt),
            },
            "f_mid": {
                "model_name": "DM_time_binary_classificator_241002_5_GAP",
                "resolution": 256,
                "mode": "ft",
                "dropout": False,
                "checkpoint": str(mid_ckpt),
            },
            "f_large": {
                "model_name": "DM_time_binary_classificator_resnet18",
                "resolution": 256,
                "mode": "dmft",
                "dropout": False,
                "checkpoint": str(large_ckpt),
            },
            "r1": {
                "model_name": "conv_mlp",
                "cnn_channels": 64,
                "extra_conv": False,
                "pool_size": 7,
                "hidden_dim": 64,
                "dropout": 0.0,
                "checkpoint": str(r1_ckpt),
            },
            "r2": {
                "model_name": "conv_mlp",
                "cnn_channels": 64,
                "extra_conv": True,
                "pool_size": 7,
                "hidden_dim": 128,
                "dropout": 0.2,
                "checkpoint": str(r2_ckpt),
            },
        },
    }


BASELINE_JOINT_CONFIG = make_joint_config(
    BASELINE_SMALL_CKPT,
    BASELINE_MID_CKPT,
    BASELINE_LARGE_CKPT,
    BASELINE_R1_CKPT,
    BASELINE_R2_CKPT,
)
FINETUNE_JOINT_CONFIG = make_joint_config(
    FINETUNE_SMALL_CKPT,
    FINETUNE_MID_CKPT,
    FINETUNE_LARGE_CKPT,
    FINETUNE_R1_CKPT,
    FINETUNE_R2_CKPT,
)


class HardThresholdJointCascade(torch.nn.Module):
    def __init__(self, cascade: torch.nn.Module, threshold_r1: float, threshold_r2: float):
        super().__init__()
        self.cascade = cascade
        self.threshold_r1 = float(threshold_r1)
        self.threshold_r2 = float(threshold_r2)
        self.reset_counts()

    def reset_counts(self):
        self.route_counts = {"small": 0, "mid": 0, "large": 0, "total": 0}

    def forward(self, batch):
        outputs = self.cascade.forward_hard_aux(
            batch,
            threshold_r1=self.threshold_r1,
            threshold_r2=self.threshold_r2,
        )
        selected = outputs["selected_expert"].detach().cpu()
        counts = torch.bincount(selected, minlength=3).tolist()
        self.route_counts["small"] += int(counts[0])
        self.route_counts["mid"] += int(counts[1])
        self.route_counts["large"] += int(counts[2])
        self.route_counts["total"] += int(selected.numel())
        return outputs["log_probs"]

    def model_distribution(self) -> dict[str, float | int]:
        counts = dict(self.route_counts)
        total = counts["total"]
        for name in ("small", "mid", "large"):
            counts[f"pi_{name}"] = counts[name] / total if total else float("nan")
        return counts


def build_baseline_rejection_ensemble() -> HardThresholdJointCascade:
    cascade = build_joint_model(BASELINE_JOINT_CONFIG, device=DEVICE).eval()
    return HardThresholdJointCascade(cascade, BASELINE_R1_THRESHOLD, BASELINE_R2_THRESHOLD).to(DEVICE).eval()


def build_finetune_rejection_ensemble() -> HardThresholdJointCascade:
    cascade = build_joint_model(FINETUNE_JOINT_CONFIG, device=DEVICE).eval()
    return HardThresholdJointCascade(cascade, FINETUNE_R1_THRESHOLD, FINETUNE_R2_THRESHOLD).to(DEVICE).eval()


def build_joint_training_rejection_ensemble() -> HardThresholdJointCascade:
    checkpoint = torch.load(JOINT_MOE_CKPT, map_location="cpu")
    config = copy.deepcopy(checkpoint["config"])
    config["dataset"]["output_dir"] = DATASET_CFG["output_dir"]
    config["dataset"]["prefix"] = DATASET_CFG["prefix"]
    cascade = build_joint_model(config, device=DEVICE)
    cascade.load_state_dict(checkpoint["model_state_dict"])
    cascade.eval()
    print(
        "Joint checkpoint geladen: "
        f"epoch={checkpoint.get('epoch')}, "
        f"val_topk/accuracy={checkpoint.get('metrics', {}).get('val_topk/accuracy', float('nan')):.4f}"
    )
    return HardThresholdJointCascade(cascade, MOE_R1_THRESHOLD, MOE_R2_THRESHOLD).to(DEVICE).eval()

In [ ]:
def compute_metrics(tp: int, fp: int, fn: int, tn: int, zero_division: float = 0.0) -> dict[str, float | int]:
    total = tp + fp + fn + tn
    accuracy = (tp + tn) / total if total else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else zero_division
    recall = tp / (tp + fn) if (tp + fn) else zero_division
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else zero_division
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "count": total,
    }


def sorted_bin_items(per_bin: dict[Any, dict[str, Any]]):
    def key_fn(item):
        key, _ = item
        return (float("inf"), 0) if key is None else (0, key)
    return sorted(per_bin.items(), key=key_fn)


def evaluate_model_by_snr(
    model: torch.nn.Module,
    dataset: DMTimeShardDataset,
    name: str,
    positive_label: int = 1,
    zero_division: float = 0.0,
) -> dict[str, Any]:
    loader_kwargs = dict(
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(NUM_WORKERS > 0),
    )
    if NUM_WORKERS > 0:
        loader_kwargs["prefetch_factor"] = 2
    loader = DataLoader(dataset, **loader_kwargs)

    if hasattr(model, "reset_counts"):
        model.reset_counts()
    model.eval()

    bin_counts = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0, "tn": 0})
    overall_counts = {"tp": 0, "fp": 0, "fn": 0, "tn": 0}

    with torch.inference_mode():
        for batch in tqdm(loader, desc=f"Evaluiere {name}"):
            snr_values = batch["metadata"][:, 0].detach().cpu().numpy()
            batch_on_device = {
                key: value.to(DEVICE, non_blocking=True) if torch.is_tensor(value) else value
                for key, value in batch.items()
            }
            labels = batch_on_device["label"]
            outputs = model(batch_on_device)
            if isinstance(outputs, dict):
                outputs = outputs["log_probs"]
            predictions = outputs.argmax(dim=1)

            labels_np = labels.detach().cpu().numpy()
            preds_np = predictions.detach().cpu().numpy()

            for snr_value, pred, label in zip(snr_values, preds_np, labels_np):
                snr_bin = int(np.rint(float(snr_value))) if np.isfinite(snr_value) else None
                pred_positive = pred == positive_label
                true_positive = label == positive_label
                if pred_positive and true_positive:
                    key = "tp"
                elif pred_positive and not true_positive:
                    key = "fp"
                elif not pred_positive and true_positive:
                    key = "fn"
                else:
                    key = "tn"
                bin_counts[snr_bin][key] += 1
                overall_counts[key] += 1

    per_bin = {}
    for snr_bin, counts in sorted_bin_items(bin_counts):
        per_bin[snr_bin] = {**compute_metrics(**counts, zero_division=zero_division), **counts}

    result = {
        "name": name,
        "split": SPLIT,
        "per_bin": per_bin,
        "overall": {**compute_metrics(**overall_counts, zero_division=zero_division), **overall_counts},
        "overall_accuracy": compute_metrics(**overall_counts, zero_division=zero_division)["accuracy"],
    }
    if hasattr(model, "model_distribution"):
        result["model_distribution"] = model.model_distribution()
    return result


def json_ready(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k) if k is not None else "non_finite": json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(v) for v in value]
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def restore_result(value: dict[str, Any]) -> dict[str, Any]:
    restored = dict(value)
    per_bin = {}
    for key, metrics in value["per_bin"].items():
        bin_key = None if key == "non_finite" else int(key)
        restored_metrics = {k: (float("nan") if v is None else v) for k, v in metrics.items()}
        per_bin[bin_key] = restored_metrics
    restored["per_bin"] = per_bin
    return restored


def save_result(result: dict[str, Any], path: Path) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(result), handle, indent=2, ensure_ascii=False)


def load_result(path: Path) -> dict[str, Any]:
    with path.open(encoding="utf-8") as handle:
        return restore_result(json.load(handle))


def plot_snr_curves(
    snr_evaluation: dict[str, Any],
    output_path: Path,
    SNR_end: float | None = None,
    SNR_start: float | None = None,
    y_min: float = 0.0,
    include_non_finite: bool = False,
):
    if "per_bin" not in snr_evaluation:
        raise ValueError("snr_evaluation must contain a 'per_bin' entry.")

    per_bin = dict(snr_evaluation["per_bin"])
    if not per_bin:
        raise ValueError("snr_evaluation['per_bin'] is empty; nothing to plot.")

    if SNR_start is not None or (SNR_end is not None and SNR_end > 0):
        filtered = {}
        for key, metrics in per_bin.items():
            if key is None:
                filtered[key] = metrics
                continue
            if SNR_start is not None and key < SNR_start:
                continue
            if SNR_end is not None and SNR_end > 0 and key > SNR_end:
                continue
            filtered[key] = metrics
        per_bin = filtered

    metric_names = ["accuracy", "precision", "recall", "f1"]
    metric_labels = {
        "accuracy": "Genauigkeit",
        "precision": "Precision",
        "recall": "Recall",
        "f1": "F1-Score",
    }
    style_cycle = {
        "accuracy": {"marker": "o", "linestyle": "-", "color": "#1f77b4"},
        "precision": {"marker": "s", "linestyle": "--", "color": "#ff7f0e"},
        "recall": {"marker": "D", "linestyle": "-.", "color": "#2ca02c"},
        "f1": {"marker": "^", "linestyle": ":", "color": "#d62728"},
    }

    snr_values = []
    metrics_by_name = {name: [] for name in metric_names}
    for snr_value, metrics in sorted_bin_items(per_bin):
        if snr_value is None and not include_non_finite:
            continue
        snr_label = -1 if snr_value is None else snr_value
        snr_values.append(snr_label)
        for name in metric_names:
            metrics_by_name[name].append(metrics.get(name, float("nan")))

    if not snr_values:
        raise ValueError("No SNR bins left to plot. Enable include_non_finite to show None bin.")

    fig, ax = plt.subplots(figsize=(7, 5))
    for name in metric_names:
        ax.plot(
            snr_values,
            metrics_by_name[name],
            label=metric_labels[name],
            **style_cycle[name],
        )

    ax.set_xlabel("SNR", fontsize=13)
    ax.set_ylabel("Metrik", fontsize=13)
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0))
    ax.set_ylim(y_min, 1.02)
    ax.set_xlim(min(snr_values), max(snr_values))
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(loc="lower right", fontsize=11)
    plt.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.show()
    return fig, ax

In [ ]:
MODEL_SPECS = [
    {
        "name": "f_small",
        "slug": "f_small",
        "pdf": "snr_metrics_f_small_test.pdf",
        "loader": lambda: build_classifier(
            "DM_time_binary_classificator_241002_3_GAP",
            "dmt",
            BASELINE_SMALL_CKPT,
            dropout=False,
        ),
    },
    {
        "name": "f_mid",
        "slug": "f_mid",
        "pdf": "snr_metrics_f_mid_test.pdf",
        "loader": lambda: build_classifier(
            "DM_time_binary_classificator_241002_5_GAP",
            "ft",
            BASELINE_MID_CKPT,
            dropout=False,
        ),
    },
    {
        "name": "f_large",
        "slug": "f_large",
        "pdf": "snr_metrics_f_large_test.pdf",
        "loader": lambda: build_classifier(
            "DM_time_binary_classificator_resnet18",
            "dmft",
            BASELINE_LARGE_CKPT,
            dropout=False,
        ),
    },
    {
        "name": "baseline_rejection_ensemble",
        "slug": "baseline_rejection_ensemble",
        "pdf": "snr_metrics_baseline_rejection_ensemble_test.pdf",
        "loader": build_baseline_rejection_ensemble,
    },
    {
        "name": "finetuned_rejection_ensemble",
        "slug": "finetuned_rejection_ensemble",
        "pdf": "snr_metrics_finetuned_rejection_ensemble_test.pdf",
        "loader": build_finetune_rejection_ensemble,
    },
    {
        "name": "joint_training_rejection_ensemble",
        "slug": "joint_training_rejection_ensemble",
        "pdf": "snr_metrics_joint_training_rejection_ensemble_test.pdf",
        "loader": build_joint_training_rejection_ensemble,
    },
]

print("Konfigurationen:")
for spec in MODEL_SPECS:
    print(f"- {spec['name']} -> {PLOT_DIR / spec['pdf']}")

In [ ]:
test_dataset = make_test_dataset()
print(f"Test samples: {len(test_dataset):,}")

all_results = {}
for spec in MODEL_SPECS:
    name = spec["name"]
    cache_path = METRIC_DIR / f"{spec['slug']}_test_metrics.json"
    pdf_path = PLOT_DIR / spec["pdf"]

    if cache_path.exists() and not RECOMPUTE_METRICS:
        print(f"\nLade Cache fuer {name}: {cache_path}")
        result = load_result(cache_path)
    else:
        print(f"\nLade und evaluiere {name}")
        model = spec["loader"]()
        result = evaluate_model_by_snr(model, test_dataset, name=name)
        save_result(result, cache_path)
        del model
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()

    plot_snr_curves(
        result,
        pdf_path,
        SNR_end=SNR_END,
        SNR_start=SNR_START,
        y_min=Y_MIN,
        include_non_finite=INCLUDE_NON_FINITE,
    )
    all_results[name] = result
    overall = result["overall"]
    print(
        f"{name}: acc={overall['accuracy']:.4f}, precision={overall['precision']:.4f}, "
        f"recall={overall['recall']:.4f}, f1={overall['f1']:.4f}, n={overall['count']:,}"
    )
    if "model_distribution" in result:
        print("Routing:", result["model_distribution"])

summary_path = PLOT_DIR / "snr_metrics_test_summary.json"
save_result({"results": all_results}, summary_path)
print(f"\nFertig. PDFs und Metriken gespeichert in: {PLOT_DIR}")